# A Developer Guide to Disciplined Trading

**Notebook 1 of a series — Foundations: techtrade + Analysis end-to-end POC**

> *"I've been trading for a year on hunches and I'm tired of losing
> to my own biases. I can read Python — can a deterministic engine
> do better than my gut?"*

---

## What this notebook is

A **single, self-contained proof-of-concept** that walks a
programming-literate newbie trader through every working command
in the OpenBB-techtrade fork as of **2026-06-27**. By the end,
you will have:

1. A working `.venv_win` environment with an editable techtrade install.
2. A fundamental health-check on one ticker via the `Analysis` 7-phase pipeline.
3. A technical setup for that ticker via `obb.techtrade.signals` / `plan` / `scan`.
4. An Excel workbook ready to print (`obb.techtrade.export`).
5. A statistical robustness verdict on the plan (`obb.techtrade.validate`).
6. An honest understanding of what is **NOT** yet shipped (tuning, narrator, MCP).

**Time budget:** 15 minutes if you only read; ~1 hour if you run
every cell (one cell is a 5-10 minute backtest validation).

---

## Who this is for — the user story

You are **Alex**, a software engineer who started day-trading during
the 2024-25 bull run, made some money, gave most of it back, and now
wants the discipline of a system rather than the gambler's high of a
hunch. You:

- Read Python well enough to debug a stack trace and refactor a function.
- Know what a *moving average* and *RSI* are by name, but couldn't
  defend a specific period choice in court.
- Want to **measure** whether a setup has an edge before you size
  into it, not after.
- Have **never** trusted a "buy this stock" indicator and don't intend
  to start — but you'd trust a *gated* recommendation that came with
  a Probability of Backtest Overfitting number attached.

This notebook hands you that pipeline. You don't get to skip the
verification — every recommendation it produces is shown alongside
its statistical caveats.

## What this notebook is NOT

- **Not financial advice.** Every output the engine produces is
  research / paper-trading material. Real-money decisions are yours.
- **Not a backtesting framework.** Backtesting is delegated to the
  separate `openbb-backtest` extension (which ships in-tree); this
  notebook calls it via `obb.techtrade.validate` but does not
  re-implement WFO / CPCV / PBO / DSR.
- **Not a replacement for fundamentals.** The `Analysis` 7-phase
  pipeline covered in §3 is the fundamental complement to
  techtrade's purely-technical view. A complete decision uses both.

---

> **Working with this notebook.** This `.ipynb` is the source of truth — edit cells directly. There is no separate builder script to keep in sync. Outputs are not committed (see `nbstripout` in the pre-commit hook), so the diff of the notebook is the diff of the prose + code only.


## 1. Environment Setup

### 1.1 Repository + virtual environment

**Windows / PowerShell** (the canonical dev setup for this fork — see
`CLAUDE.md`):

```powershell
# 1. Clone the fork
git clone https://github.com/prajoria/OpenBB.git OpenBBTechnical
cd OpenBBTechnical
git checkout trading_technicals

# 2. Create the project venv (Python 3.10-3.13; this fork is dev'd on 3.12)
py -3.12 -m venv .venv_win

# 3. Activate
.\.venv_win\Scripts\Activate.ps1

# 4. Editable install of every extension
cd openbb_platform
python dev_install.py -e
cd ..

# 5. Install the notebook-only runtime extras (Jupyter, pyarrow, openpyxl, ...)
python -m pip install -r notebooks/requirements.txt

# 6. Launch
jupyter lab notebooks/01-foundations-techtrade-and-analysis.ipynb
```

**macOS / Linux:** identical except `.venv_win` becomes `.venv` and
the activation path is `source .venv/bin/activate`. The rest of the
commands are identical; paths shown in this notebook use `.venv_win`
because the primary dev environment is Windows.

> **Step 5 — `notebooks/requirements.txt`** holds the pip packages the
> notebooks import directly but that the editable install in step 4 does
> not guarantee: `jupyterlab`, `ipykernel`, **`pyarrow`** (the parquet
> engine `obb.techtrade.validate` needs to write backtest bundles in §5),
> `openpyxl` (Excel export in §4), and `python-dotenv`. If §5 ever raises
> `Unable to find a usable engine ... pyarrow or fastparquet`, you skipped
> this step — run it and restart the kernel.

### 1.2 Credentials — `fmp_cached` is the only provider

This fork is hard-pinned to the `fmp_cached` provider (a caching
wrapper around FinancialModelingPrep) — there is no `yfinance`
fallback. Configure once in
**`~/.openbb_platform/user_settings.json`**:

```json
{
  "credentials": {
    "fmp_api_key": "YOUR_FMP_KEY",
    "fmp_cached_api_key": "YOUR_FMP_KEY"
  }
}
```

> Both keys point to the same FMP API key — `fmp_cached` reuses
> `fmp`'s credential. You can get a free starter key at
> <https://financialmodelingprep.com/>; the free tier covers
> everything in this notebook.

**Optional**: drop a `.env` at the repo root with `FMP_API_KEY=...`
for scripts that load it via `python-dotenv`.

> **Privacy:** `.env` and `user_settings.json` are both `.gitignore`d.
> Never commit them — even if your fork is private. Quants who have
> pushed an FMP key to a public repo have had it rotated by FMP
> within hours.

### 1.3 What's installed where

Run the cell below to verify your environment is wired correctly.
All of these are **required**:

In [14]:
# Cell 1.3 — environment check (no API calls; safe to run repeatedly).
import importlib.util
import sys

GREEN  = "\033[92m"
RED    = "\033[91m"
YELLOW = "\033[93m"
RESET  = "\033[0m"

REQUIRED = [
    "openbb",                  # Top-level OpenBB
    "openbb_techtrade",        # This fork's technical-trading extension
    "openbb_backtest",         # Validation engine (#82 dep)
    "openbb_fmp_cached",       # The only provider this fork uses
    "openpyxl",                # Excel export engine (§4)
    "pandas_ta_classic",       # Vendored indicator library
    "pyarrow",                 # Parquet engine for validate bundles (§5)
]
OPTIONAL = [
    ("tuneta",  "needed only for obb.techtrade.tune (#83); pip install 'openbb-techtrade[tuneta]'"),
    ("xlsxwriter", "richer Excel engine; pip install 'openbb-techtrade[xlsxwriter]'"),
]

print(f"Python: {sys.version.split()[0]}  ({sys.executable})\n")
print("Required packages:")
missing = []
for pkg in REQUIRED:
    ok = importlib.util.find_spec(pkg) is not None
    tag = f"{GREEN}OK {RESET}" if ok else f"{RED}XX {RESET}"
    print(f"  {tag} {pkg}")
    if not ok:
        missing.append(pkg)

print("\nOptional extras (notebook works without them):")
for pkg, hint in OPTIONAL:
    ok = importlib.util.find_spec(pkg) is not None
    tag = f"{GREEN}OK {RESET}" if ok else f"{YELLOW}-- {RESET}"
    print(f"  {tag} {pkg:<14} {'' if ok else hint}")

if missing:
    raise RuntimeError(
        f"Missing required packages: {missing}. "
        "Run `python -m pip install -r notebooks/requirements.txt` (and, for the "
        "openbb_* packages, `cd openbb_platform && python dev_install.py -e`) from .venv_win."
    )
print(f"\n{GREEN}Environment OK.{RESET}")

Python: 3.12.13  (h:\masterswork\git\OpenBBTechnical\.venv_win\Scripts\python.exe)

Required packages:
  OK  openbb
  OK  openbb_techtrade
  OK  openbb_backtest
  OK  openbb_fmp_cached
  OK  openpyxl
  OK  pandas_ta_classic
  OK  pyarrow

Optional extras (notebook works without them):
  --  tuneta         needed only for obb.techtrade.tune (#83); pip install 'openbb-techtrade[tuneta]'
  --  xlsxwriter     richer Excel engine; pip install 'openbb-techtrade[xlsxwriter]'

Environment OK.


**Expected output (yours may differ on optionals):**

```
Python: 3.12.13  (H:\masterswork\git\OpenBBTechnical\.venv_win\Scripts\python.exe)

Required packages:
  OK  openbb
  OK  openbb_techtrade
  OK  openbb_backtest
  OK  openbb_fmp_cached
  OK  openpyxl
  OK  pandas_ta_classic

Optional extras (notebook works without them):
  --  tuneta         needed only for obb.techtrade.tune (#83); pip install 'openbb-techtrade[tuneta]'
  OK  xlsxwriter

Environment OK.
```

If any **required** package is missing, the rest of the notebook
will not work — go back to §1.1 and re-run `dev_install.py -e`.


In [2]:
# Cell 1.4 — credentials probe.  Auto-seeds user_settings.json from .env if keys are missing.
import json, os, re
from pathlib import Path

GREEN  = "\033[92m"
RED    = "\033[91m"
YELLOW = "\033[93m"
BOLD   = "\033[1m"
RESET  = "\033[0m"

settings_path = Path.home() / ".openbb_platform" / "user_settings.json"

# ------------------------------------------------------------------
# Auto-seed from .env if settings file is missing or keys are unset
# ------------------------------------------------------------------
def _key_from_env_file(repo_root: Path) -> str | None:
    """Read FMP_API_KEY from .env in the repo root (never from os.environ)."""
    env_file = repo_root / ".env"
    if not env_file.exists():
        return None
    for line in env_file.read_text(encoding="utf-8").splitlines():
        m = re.match(r"^FMP_API_KEY\s*=\s*(.+)$", line.strip())
        if m:
            return m.group(1).strip()
    return None

def _seed_settings(settings_path: Path, key: str) -> None:
    """Write key into user_settings.json, creating the file if needed."""
    settings_path.parent.mkdir(parents=True, exist_ok=True)
    data = {}
    if settings_path.exists():
        try:
            data = json.loads(settings_path.read_text(encoding="utf-8"))
        except json.JSONDecodeError:
            data = {}
    creds = data.setdefault("credentials", {})
    creds["fmp_api_key"] = key
    creds["fmp_cached_api_key"] = key
    settings_path.write_text(json.dumps(data, indent=2), encoding="utf-8")

# Locate repo root (walk up from notebook location)
repo_root = Path(__file__).resolve().parent if "__file__" in dir() else Path.cwd().resolve()
while not (repo_root / ".env").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent

creds_now = {}
if settings_path.exists():
    creds_now = json.loads(settings_path.read_text(encoding="utf-8")).get("credentials", {})

has_fmp_cached = bool(creds_now.get("fmp_cached_api_key"))
has_fmp        = bool(creds_now.get("fmp_api_key"))

if not (has_fmp_cached or has_fmp):
    key = _key_from_env_file(repo_root)
    if key:
        _seed_settings(settings_path, key)
        print(f"{YELLOW}Auto-seeded{RESET} credentials from .env → {settings_path}")
        creds_now = json.loads(settings_path.read_text(encoding="utf-8")).get("credentials", {})
        has_fmp_cached = bool(creds_now.get("fmp_cached_api_key"))
        has_fmp        = bool(creds_now.get("fmp_api_key"))
    else:
        print(f"{RED}No .env found at {repo_root}/.env{RESET}")

# ------------------------------------------------------------------
# Report
# ------------------------------------------------------------------
print(f"Credentials file: {settings_path}\n")
tag_cached = f"{GREEN}OK{RESET}" if has_fmp_cached else f"{RED}MISSING{RESET}"
tag_fmp    = f"{GREEN}OK{RESET}" if has_fmp        else f"{RED}MISSING{RESET}"
print(f"  fmp_cached_api_key : {tag_cached}")
print(f"  fmp_api_key        : {tag_fmp}")

if not (has_fmp_cached or has_fmp):
    print(f"\n{RED}{BOLD}Action required:{RESET} add your FMP key to {settings_path}")
    print('  "credentials": { "fmp_api_key": "KEY", "fmp_cached_api_key": "KEY" }')
    print("  Or drop FMP_API_KEY=... in a .env at the repo root.")
    raise RuntimeError("No FMP credentials configured. See instructions above.")

print(f"\n{GREEN}Credentials OK.{RESET}")


Credentials file: C:\Users\daaji\.openbb_platform\user_settings.json

  fmp_cached_api_key : OK
  fmp_api_key        : OK

Credentials OK.


### 1.5 The `obb` object

All techtrade commands hang off the top-level `obb` object. Importing
it once at the top of the notebook makes every later cell a one-liner.
The first import is slow (~5-10 seconds) — that's `openbb.build()`
generating the typed Python surface from every installed extension's
entry points. Later imports are instant.


In [3]:
# Cell 1.5 — load OpenBB. First call is slow; later cells are instant.
from openbb import obb
techtrade_cmds = sorted(
    c for c in dir(obb.techtrade)
    if not c.startswith("_") and callable(getattr(obb.techtrade, c, None))
)
print(f"obb loaded. {len(techtrade_cmds)} commands under obb.techtrade.*:")
for cmd in techtrade_cmds:
    print(f"  obb.techtrade.{cmd}")

obb loaded. 11 commands under obb.techtrade.*:
  obb.techtrade.about
  obb.techtrade.export
  obb.techtrade.movers
  obb.techtrade.orders
  obb.techtrade.plan
  obb.techtrade.scan
  obb.techtrade.segments
  obb.techtrade.signals
  obb.techtrade.simulate
  obb.techtrade.tune
  obb.techtrade.validate


**Expected:** the printout should include `about`, `export`, `tune`,
`movers`, `orders`, `plan`, `scan`, `segments`, `signals`,
`simulate`, `validate`. If `tune` is missing, that's correct —
it's not yet shipped (see §6 Roadmap).


### 1.6 Who holds this stock, and how much? - SEC bulk 13F CUSIP index (#89)

[Form 13F-HR](https://www.investopedia.com/terms/f/form-13f.asp) ([SEC FAQ](https://www.sec.gov/divisions/investment/13ffaq.htm)) is filed *by* institutional managers and indexed by the
**filer**, so it cannot answer "which managers hold `MSFT`?" directly.
`Tools/ingest_sec_13f.py` loads SEC's quarterly **bulk** 13F data set into a
local **[CUSIP](https://www.investopedia.com/terms/c/cusipnumber.asp)-keyed** index, and the read helpers invert the question:
`ticker -> CUSIP(s) -> holders`. The same index powers the `fmp_cached`
institutional-ownership SEC fallback tier (`_try_sec_13f`).

This cell shows three things: the **aggregate institutional ownership %**, the
manager count, and the **top-5 holders**. Two helpers do the work —
`institutional_holding_summary()` sums every reporting manager's position for
the stock's CUSIP in one quarter, and `holders_for_cusip()` returns the ranked
holder list.

**Authentic sources.** The numerator (institutional shares) comes from SEC's
bulk 13F data set. The denominator (shares outstanding) comes from **[SEC EDGAR](https://www.investopedia.com/terms/e/edgar.asp)'s
[XBRL company-facts API](https://www.sec.gov/structureddata)** — the issuer's own cover-page count
(`dei:EntityCommonStockSharesOutstanding`), period-aligned to the 13F quarter so
both figures come from the same regulator. `shares_outstanding_from_sec()`
fetches it on demand; if SEC is unreachable the cell falls back to the
`fmp_cached` profile (a reputable third-party aggregator).

**Expected:** Microsoft's CUSIP `594918104`, ~75% institutional ownership for
the 2023-Q1 snapshot, and a ranked list led by Vanguard / BlackRock / State
Street. If the index is empty, the cell prints a hint instead of failing -
graceful degradation, never an exception.

> **Snapshot caveat.** 13F is a quarter-end snapshot filed up to 45 days late,
> and SEC's bulk endpoint currently serves through **2023-Q2** (reporting
> 2023-Q1 positions), so absolute figures reflect mid-2023, not today. The
> ranking and ownership % are representative; the exact numbers will drift from
> live quotes. Once **issue #99 (SEC N-PORT ingest)** ships, the same pattern extends to ETF holdings via Form N-PORT-P (see §6 roadmap).


In [4]:
# Cell 1.6 - SEC bulk 13F CUSIP index smoke check (#89)
# "Which institutional managers hold this stock, and how much of it do they own?"
# Form 13F is filed BY managers and indexed by the filer, so it cannot answer
# this directly. The local CUSIP index (built by Tools/ingest_sec_13f.py) inverts
# it: ticker -> CUSIP -> holders. This same index powers the fmp_cached
# institutional-ownership SEC fallback tier.
#
# For the institutional OWNERSHIP % we also need shares outstanding. The most
# authentic source is SEC EDGAR's XBRL company-facts API (the issuer's own
# cover-page count) - same regulator as the 13F data, so numerator and
# denominator agree and period-align. fmp_cached is used as a secondary source
# if the SEC figure is unavailable.
from openbb_sec.utils.thirteen_f_index import (
    holders_for_cusip,
    institutional_holding_summary,
    resolve_cusip,
    shares_outstanding_from_sec,
)

SYMBOL = "MSFT"
cusips = resolve_cusip(SYMBOL)
print(f"{SYMBOL} -> CUSIP(s): {cusips}")

# Latest period present in the index for this stock (drives both queries below).
period = None
holders = holders_for_cusip(cusips)[:5] if cusips else []
if holders:
    period = holders[0]["period"]

# --- Shares outstanding: authentic SEC EDGAR first, fmp_cached as fallback ----
shares_out = shares_outstanding_from_sec(SYMBOL, period) if cusips else None
shares_src = "SEC EDGAR (dei:EntityCommonStockSharesOutstanding)"
if not shares_out:
    try:
        from openbb import obb
        prof = obb.equity.profile(symbol=SYMBOL, provider="fmp_cached").results
        shares_out = getattr(prof[0], "shares_outstanding", None) if prof else None
        shares_src = "fmp_cached profile (fallback)"
    except Exception as exc:  # noqa: BLE001
        print(f"  (fmp_cached shares-outstanding fallback unavailable: {exc})")

# --- Aggregate institutional ownership % -------------------------------------
summary = institutional_holding_summary(cusips, period=period, shares_outstanding=shares_out)
if summary["holder_count"]:
    pct = summary["pct_institutional"]
    pct_str = f"{pct * 100:.1f}%" if pct is not None else "n/a (shares outstanding unknown)"
    print(f"\nInstitutional ownership of {SYMBOL} (period {summary['period']}):")
    print(f"  Reporting managers (13F): {summary['holder_count']:,}")
    print(f"  Institutional shares:     {summary['total_shares']:>15,}")
    print(f"  Institutional value:      ${summary['total_value_usd']:>15,}")
    if shares_out:
        print(f"  Shares outstanding:       {shares_out:>15,}   [{shares_src}]")
    print(f"  % held by institutions:   {pct_str}")
else:
    print("No holdings in the local index yet -- run Tools/ingest_sec_13f.py to populate it.")

# --- Top-5 managers by reported value ----------------------------------------
if holders:
    print(f"\nTop {len(holders)} managers holding {SYMBOL} (period {period}):")
    for h in holders:
        name = (h["filer_name"] or "")[:40]
        print(f"  {name:40s} shares={h['shares']:>12,}  value_usd=${h['value_usd']:>15,}")


MSFT -> CUSIP(s): ['594918104']

Institutional ownership of MSFT (period 2023-Q1):
  Reporting managers (13F): 4,515
  Institutional shares:       5,586,915,374
  Institutional value:      $1,384,457,411,802
  Shares outstanding:         7,435,487,575   [SEC EDGAR (dei:EntityCommonStockSharesOutstanding)]
  % held by institutions:   75.1%

Top 5 managers holding MSFT (period 2023-Q1):
  VANGUARD GROUP INC                       shares= 649,516,597  value_usd=$187,255,634,916
  BlackRock Inc.                           shares= 537,573,096  value_usd=$154,982,323,597
  STATE STREET CORP                        shares= 292,106,885  value_usd=$ 84,214,184,961
  FMR LLC                                  shares= 200,523,373  value_usd=$ 57,810,888,675
  JPMORGAN CHASE & CO                      shares= 185,732,072  value_usd=$ 54,556,590,392


## 2. Alex's Workflow — what we're going to actually do

Alex's research process for one trading day looks like this:

```mermaid
flowchart TD
    A["Pick which stocks to consider<br/>(the 'universe')"] --> B["Fundamentals check<br/>§3 Analysis 7-phase pipeline"]
    B -- "is the company healthy<br/>enough to even trade?" --> C["Technical setup<br/>§4 obb.techtrade.signals / plan / scan"]
    C -- "what's the signal,<br/>where do stops go?" --> D["Excel export<br/>§4.4 obb.techtrade.export"]
    D -- "printable plan for<br/>the trading day" --> E["Robustness gate<br/>§5 obb.techtrade.validate"]
    E -- "would this rule have<br/>survived out-of-sample?" --> F["Decision — Alex's call<br/>(the machine never auto-trades)"]

    classDef universe fill:#2563eb,stroke:#1e3a8a,stroke-width:2px,color:#ffffff;
    classDef fundamentals fill:#7c3aed,stroke:#4c1d95,stroke-width:2px,color:#ffffff;
    classDef technical fill:#0891b2,stroke:#155e75,stroke-width:2px,color:#ffffff;
    classDef export fill:#15803d,stroke:#14532d,stroke-width:2px,color:#ffffff;
    classDef gate fill:#b45309,stroke:#7c2d12,stroke-width:2px,color:#ffffff;
    classDef decision fill:#b91c1c,stroke:#7f1d1d,stroke-width:2px,color:#ffffff;

    class A universe;
    class B fundamentals;
    class C technical;
    class D export;
    class E gate;
    class F decision;
```

Alex uses **one ticker for the deep-dive** (Microsoft, `MSFT` — large
cap, liquid, well-understood — picked because every public study
covers it so Alex can sanity-check) and **one segment for the scan**
(Information Technology, the GICS sector MSFT belongs to).

> **NG4 — Non-goal #4 from the PRD:** the agent layer never makes
> trading decisions. Every command in this notebook is a research
> tool. Alex pulls the trigger, not the engine.

## 3. Fundamentals — the `Analysis` 7-phase pipeline

Before Alex looks at a chart, the company needs to clear basic
financial-health bars. The `Analysis/stock_analysis.py` module ships
a standalone, deterministic 7-phase pipeline that scores one ticker
from company profile through final decision. It uses **only**
`fmp_cached` (no live FMP calls if your cache is warm).

**The 7 phases** (full glossary in `Analysis/docs/FINANCIAL_DOMAIN_GLOSSARY.md`):

| Phase | What it scores | Why Alex cares |
|---|---|---|
| P1 | Company profile + market cap + sector | Is this even tradeable? Is it a microcap nobody can exit? |
| P2 | Fundamentals (revenue, margins, debt) | Is the underlying business solvent? |
| P3 | Technicals (price trend, volume) | Is price action sensible? |
| P4 | Valuation ([DCF](https://www.investopedia.com/terms/d/dcf.asp) [margin of safety](https://www.investopedia.com/terms/m/marginofsafety.asp)) | Is it priced as if everything will go right? |
| P5 | Risk (volatility, [beta](https://www.investopedia.com/terms/b/beta.asp), [drawdowns](https://www.investopedia.com/terms/d/drawdown.asp)) | What's my downside if I'm early? |
| P6 | Peer relative | How does it stack vs sector peers? |
| P7 | Composite decision (BUY/SELL/HOLD + score) | The summary line |


In [5]:
# Cell 3.1 — run the 7-phase analysis on MSFT.
# Wall-clock: ~30-90 seconds on a warm fmp_cached cache; longer on first run.
# The pipeline reads obb.user.credentials.fmp_cached_api_key automatically
# because we configured user_settings.json in §1.2.
import contextlib
import io
import logging
import re
import sys
from pathlib import Path

# Make the standalone Analysis module importable from the repo root.
repo_root = Path.cwd().resolve()
while not (repo_root / "Analysis").is_dir() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
if (repo_root / "Analysis").is_dir():
    sys.path.insert(0, str(repo_root / "Analysis"))
else:
    raise RuntimeError(
        "Could not find Analysis/ directory. Are you in the repo? "
        f"Started from {Path.cwd()}"
    )

from stock_analysis import AnalysisConfig, run_full_analysis

# --- Quiet the benign institutional-ownership fallback noise --------------
# obb.equity.ownership.institutional walks a FMP -> yfinance -> SEC fallback
# chain. On a restricted FMP subscription tier the first tiers fail HARMLESSLY
# and emit warnings that look alarming but are not fatal:
#   * "FMP institutional ownership failed: ... 402 Restricted Endpoint"
#   * yfinance "HTTP Error 401: ... Invalid Crumb" / "unable to access feature"
#   * "_to_df failed: Results not found."
# The pipeline still returns all seven phases (p1..p7) — institutional
# ownership is an optional Phase-1 input that simply comes back empty here.
# We raise these loggers to ERROR and filter the matching stderr lines so the
# tutorial output stays readable. Remove this block if you want to see them.
for _noisy in ("stock_analysis", "openbb_fmp_cached", "yfinance"):
    logging.getLogger(_noisy).setLevel(logging.ERROR)

_BENIGN = re.compile(
    r"institutional ownership failed|Invalid Crumb|"
    r"unable to access this feature|_to_df failed|HTTP Error 401",
    re.IGNORECASE,
)


class _StderrFilter(io.TextIOBase):
    """Drop only known-benign fallback lines; pass everything else through."""

    def __init__(self, target):
        self._target = target

    def write(self, text):  # noqa: D102
        if not _BENIGN.search(text):
            self._target.write(text)
        return len(text)

    def flush(self):  # noqa: D102
        self._target.flush()


cfg = AnalysisConfig(symbol="MSFT")
with contextlib.redirect_stderr(_StderrFilter(sys.stderr)):
    results = run_full_analysis(cfg)
print(f"Pipeline phases returned: {sorted(results.keys())}")


Pipeline phases returned: ['p1', 'p2', 'p3', 'p4', 'p5', 'p6', 'p7']


> **What you should see:** something like
> `Pipeline phases returned: ['p1', 'p2', 'p3', 'p4', 'p5', 'p6', 'p7']`.
> Each key holds a typed dataclass — `results['p7']` carries the
> final action label and composite score.
>
> **If you see a `RuntimeError: obb is unavailable`** — your
> `user_settings.json` doesn't have a valid `fmp_cached_api_key`
> (go back to §1.2).
>
> **About the suppressed warnings.** Phase 1 asks for *institutional
> ownership* via `obb.equity.ownership.institutional`, which walks an upstream-OpenBB
> `FMP → [yfinance](https://pypi.org/project/yfinance/) → SEC` fallback chain. On a restricted FMP plan you'd
> otherwise see harmless noise like `402 Restricted Endpoint`,
> yfinance `HTTP Error 401: Invalid Crumb`, and `_to_df failed: Results
> not found.`. These are **non-fatal** — institutional ownership is an
> *optional* input, so even if every tier misses, the pipeline still
> returns all seven phases. Cell 3.1 quiets these specific messages; delete
> that suppression block if you want to inspect them.
>
> **Why the *naive* SEC lookup returns nothing — and how #89 fixed it.** The
> chain *does* reach SEC — it isn't skipped. The problem is one of *indexing*:
> **Form 13F-HR is filed *by* institutional managers to report the securities
> they hold, and is indexed by the *filer* (manager CIK), not by the *held*
> company's ticker.** An operating company like MSFT never files a 13F-HR, so
> the old lookup keyed on `symbol="MSFT"` legitimately found zero filings. To
> answer "who holds MSFT" you must instead scan 13F holdings across *all*
> managers and filter by MSFT's **CUSIP**. **Issue #89 built exactly that** —
> a local CUSIP-keyed reverse index (`Tools/ingest_sec_13f.py`) that the
> `fmp_cached` SEC tier now queries. The **bonus cell below demonstrates it**
> returning real MSFT holders. (Whether it populates during *this* pipeline
> run depends on whether your active cache database holds the index — it is
> loaded into `openbb_fmp_cache_test`.)


#### Bonus: the working SEC tier, demonstrated (issue #89)

The note above explains why the institutional-ownership fallback's **SEC tier
returns nothing when you query Form 13F by the held ticker** — 13F-HR is indexed
by the *filer* (manager CIK), not by the *held* company. Issue #89 fixed this
by building a local **CUSIP-keyed reverse index** (`Tools/ingest_sec_13f.py` →
`sec_13f_holdings` / `sec_13f_cusip_map`).

The cell below calls `_try_sec_13f(["MSFT"])` — the **exact function the
`fmp_cached` provider uses** for its SEC fallback tier. It now inverts
`ticker → CUSIP → 13F holders` and returns a real, normalized ownership summary
row, proving the tier works end-to-end.

> **What you should see:** a row for `MSFT` with **~4,500 investors holding** and
> **~5.6 billion 13F shares** (period 2023-Q1), worth well over \$1 trillion —
> i.e. the SEC data the old by-ticker lookup could never surface.
>
> **Trade lesson — know your data's *grain*.** The same SEC filing answers
> opposite questions depending on how it's indexed. "Who does manager X hold?"
> (filer-keyed) and "Who holds stock Y?" (CUSIP-keyed) need *different* indexes
> over the *same* rows. Picking the wrong grain makes good data look empty.


In [6]:
# Cell 3.1b - demo the *working* SEC tier (issue #89).
# _try_sec_13f is the EXACT function fmp_cached calls for its SEC institutional-
# ownership fallback tier. The naive "Form 13F by ticker" lookup always returns
# empty (13F is indexed by the filer/manager CIK, not the held company). This
# tier instead inverts ticker -> CUSIP -> 13F holders via the local CUSIP index
# (Tools/ingest_sec_13f.py), so the tier that "returns nothing by ticker" now
# yields a real, normalized ownership summary row.
from openbb_fmp_cached.models.institutional_ownership import _try_sec_13f

# _try_sec_13f is async; top-level await works in a Jupyter cell.
sec_rows = await _try_sec_13f(["MSFT"])

if sec_rows:
    row = sec_rows[0]
    print("SEC 13F fallback tier returned a normalized ownership row "
          "(data_source='sec_13f'):\n")
    print(f"  Symbol:               {row['symbol']}")
    print(f"  As-of (quarter end):  {row['date']}")
    print(f"  Investors holding:    {row['investors_holding']:,}")
    print(f"  13F shares reported:  {row['number_of_13f_shares']:,}")
    print(f"  Total invested (USD): ${row['total_invested']:,.0f}")
    print("\nThis is the same tier obb.equity.ownership.institutional reaches "
          "after the\nFMP and yfinance tiers - now backed by the local CUSIP "
          "index instead of the\nalways-empty filer-indexed 13F-HR lookup.")
else:
    print("SEC 13F tier returned nothing - is the local index populated?")
    print("Run Tools/ingest_sec_13f.py to load it (see Tools/docs/DESIGN.md).")


SEC 13F fallback tier returned a normalized ownership row (data_source='sec_13f'):

  Symbol:               MSFT
  As-of (quarter end):  2023-03-31
  Investors holding:    4,515
  13F shares reported:  5,586,915,374
  Total invested (USD): $1,384,457,411,802

This is the same tier obb.equity.ownership.institutional reaches after the
FMP and yfinance tiers - now backed by the local CUSIP index instead of the
always-empty filer-indexed 13F-HR lookup.


In [7]:
# Cell 3.2 — inspect the Phase-7 decision and WHY it landed there.
p7 = results["p7"]
print(f"Symbol:          {cfg.symbol}")
print(f"Action label:    {p7.action_label}")
print(f"Composite score: {p7.composite_score:.2f} / 5.0")

# Label thresholds (see _decision_label in Analysis/stock_analysis.py):
#   >= 4.2 Strong Buy | >= 3.6 Buy | >= 2.8 Hold/Watch | else Avoid
# The composite is a weighted blend of six block scores. These weights
# mirror run_phase7(); keep them in sync if the pipeline changes.
weights = {
    "business_quality": 0.08,
    "fundamentals":     0.25,
    "technicals":       0.15,
    "valuation":        0.20,
    "risk_fit":         0.12,
    "peer_relative":    0.20,
}
print("\nWhy — score breakdown (block score x weight = contribution):")
for block, raw in p7.score_breakdown.items():
    w = weights.get(block, 0.0)
    print(f"  {block:<18} {raw:>4.2f} / 5  x {w:0.2f}  = {raw * w:>5.2f}")

# Hard overrides are the usual reason a strong company still reads "Avoid":
# one red flag (distress Altman Z, bearish weekly trend, high accruals) can
# cap the composite regardless of the block scores.
if p7.hard_override:
    print(f"\nHard override (caps the score): {p7.hard_override}")
else:
    print("\nNo hard override — the label reflects the weighted blend only.")


Symbol:          MSFT
Action label:    Avoid
Composite score: 2.49 / 5.0

Why — score breakdown (block score x weight = contribution):
  business_quality   4.00 / 5  x 0.08  =  0.32
  fundamentals       3.47 / 5  x 0.25  =  0.87
  technicals         1.54 / 5  x 0.15  =  0.23
  valuation          2.00 / 5  x 0.20  =  0.40
  risk_fit           1.50 / 5  x 0.12  =  0.18
  peer_relative      2.46 / 5  x 0.20  =  0.49

Hard override (caps the score):  | Weekly trend bearish — technical score capped at 2.0


### 3.3 What Alex should take from §3

You just saw a number — `2.37 / 5.0 → Avoid` — and a six-row breakdown.
This section unpacks **how each row earns its score** and **what a trader
should actually learn from it**, so the verdict stops being a black box.

#### The composite is a weighted vote, not an average

Every phase outputs a **0–5 block score**. Phase 7 blends them with fixed
weights (they live in `run_phase7()`), then maps the result to a label:

| Block (phase) | Weight | Why it carries that weight |
|---|---|---|
| `fundamentals` (P2) | **0.25** | A solvent, growing business is the foundation — biggest single vote. |
| `valuation` (P4) | **0.20** | Even a great business is a bad trade if you overpay. |
| `peer_relative` (P6) | **0.20** | Money flows to the *best* name in a sector, not just a *good* one. |
| `technicals` (P3) | 0.15 | Price action confirms or contradicts the story. |
| `risk_fit` (P5) | 0.12 | How much pain you'll eat if you're early. |
| `business_quality` (P1) | 0.08 | Tradeability / size — a low-stakes sanity check. |

`composite = Σ (block_score × weight)` → label thresholds:
**≥ 4.2 Strong Buy · ≥ 3.6 Buy · ≥ 2.8 Hold/Watch · else Avoid**
(`_decision_label`).

> **Trade lesson — diversify your evidence.** No single indicator decides a
> trade. The pipeline deliberately makes one strong signal *insufficient*:
> MSFT's great fundamentals (0.87 of the score) couldn't rescue it because
> four other blocks voted weak. Real edge comes from signals that *agree*.

#### How each block earns its 0–5 score

**P1 — business_quality** (`_score_phase1`): starts at 2.5, then adds for
size (`+0.75` mega-cap ≥ \$100B, scaling down to micro-cap), peer coverage
(`+0.5` if ≥ 5 peers), and geographic data (`+0.25`).
*Trade lesson:* this is a **liquidity / "can I get out?" gate**, not a
quality verdict. A microcap that scores low here can be impossible to exit
without moving the price against yourself.

**P2 — fundamentals** (0–5, weighted internally): revenue growth, margins,
debt load, and the **[Sloan accruals ratio](https://www.investopedia.com/terms/q/qualityofearnings.asp) ([academic source](https://en.wikipedia.org/wiki/Earnings_quality))** (how much profit is real cash vs.
accounting estimates). High accruals (> 20%) trigger a *hard cap* later.
*Trade lesson:* **earnings quality > earnings size.** A company "beating EPS"
on accruals is borrowing from next quarter — the accruals cap encodes that.

**P3 — technicals** = `bullish_count / 11 × 5`. There are 11 yes/no signals
(trend, moving-average stack, volume, momentum…). MSFT scored **0.77**, so
only ~2 of 11 fired — and the **weekly trend was bearish**, which capped this
block at 2.0.
*Trade lesson:* **don't fight the tape.** Fundamentals tell you *what* to
own; technicals tell you *when*. A bearish weekly trend means the crowd is
still selling — being "right early" loses money until price agrees.

**P4 — valuation** (`_score_phase4`): starts at 2.5; **margin of safety**
(MOS = how far price sits below DCF fair value) adds up to `+1.0` (MOS ≥ 25%)
or subtracts `0.5` when price is *above* fair value. [Piotroski](https://www.investopedia.com/terms/p/piotroski-score.asp)oski F-score and
Altman Z adjust it; **Altman Z < 1.81 hard-caps the block to 1.0** (distress).
MSFT scored **2.00** → it's priced at/above fair value (no margin of safety).
*Trade lesson:* **your entry price is your risk.** Paying up removes your
cushion — the same company is a buy at \$300 and a pass at \$450.

**P5 — risk_fit** (`_score_phase5`): Sharpe (return per unit of volatility),
Calmar (return per unit of drawdown), and max drawdown. `+1.0` for Sharpe
> 1.5, penalties for Sharpe < 0.3, Calmar < 0.5, or drawdown < −50%.
MSFT scored **1.50** → weak risk-adjusted return over this window.
*Trade lesson:* **a 20% gain that needed a 40% drawdown to get there is a bad
trade.** Position size off *risk*, not hoped-for return.

**P6 — peer_relative**: ranks the stock against its sector peers on a
quality-value basis and risk-adjusted performance (Information Ratio).
MSFT scored **2.44** → middle-of-the-pack vs. peers on this window.
*Trade lesson:* **relative strength is where institutional money goes.** If a
name lags its own sector, there's usually a better horse in the same race.

**P7 — composite + hard overrides**: the weighted blend, *then* safety caps.
Any one of these can force a low label regardless of the blend:
`Altman Z < 1.81 → ≤ 2.0` · `accruals > 20% → ≤ 2.8` · `bearish weekly trend
→ technicals ≤ 2.0` · `earnings within 5 days → technicals ≤ 2.5`.
*Trade lesson:* **risk management overrides opportunity.** One red flag
vetoes a pretty story — that's a feature, not a bug.

#### Putting MSFT together

`business_quality 4.0` and `fundamentals 3.47` say *good company*. But
`technicals 0.77` + `valuation 2.0` + `risk_fit 1.5` + `peer_relative 2.44`
say *wrong price, wrong moment, lagging peers* — and the bearish-weekly-trend
override sealed it. The verdict isn't "MSFT is bad," it's **"not here, not
now."** Re-run with a different `end_date` and the timing blocks (P3/P5) can
flip the label entirely.

#### Takeaways

- **A composite ≥ 3.5/5 with no red P5 (risk) flags** is the rough
  floor for "worth a technical setup look." Below that, even a
  beautiful chart is fighting bad fundamentals.
- **The Analysis pipeline does NOT issue trade orders.** It scores
  the *company* (and its current timing). The next section scores the *setup*.
- You can run the pipeline phase-by-phase if you want to skip
  expensive ones — see the docstring at the top of
  `Analysis/stock_analysis.py` for the per-phase function imports.


### 3.4 Is Now a Good Time to Trade This Stock?

> **Why this section exists — a note for developers**
>
> If you come from software engineering, you're used to asking:
> *"Does the function work correctly?"* In trading the question is different:
> *"Even if the function works correctly, is this a safe moment to call it?"*
>
> The 7-phase Analysis pipeline tells you whether the **company** is healthy.
> It does **not** tell you whether the **market** is in a state where acting
> on that signal is sensible. This section fills that gap.

---

#### The two questions you must separate

```mermaid
flowchart LR
    Q["Should I trade MSFT today?"]
    Q --> A["<b>§3 Fundamentals</b><br/>Is MSFT a good business?<br/>(P1..P7 pipeline)"]
    Q --> B["<b>§3.4 Market timing</b><br/>Is the market safe<br/>enough to trade it?"]
    A --> D["Trade only when<br/>BOTH answers agree"]
    B --> D

    classDef question fill:#2563eb,stroke:#1e3a8a,stroke-width:2px,color:#ffffff;
    classDef fundamentals fill:#7c3aed,stroke:#4c1d95,stroke-width:2px,color:#ffffff;
    classDef technical fill:#0891b2,stroke:#155e75,stroke-width:2px,color:#ffffff;
    classDef decision fill:#b91c1c,stroke:#7f1d1d,stroke-width:2px,color:#ffffff;
    class Q question;
    class A fundamentals;
    class B technical;
    class D decision;
```

A company can be rock-solid (P7 score 4.5/5) but if the market is
in a high-fear, high-volatility regime, the stock may drop 20% anyway
because **everyone** is selling — not because of anything MSFT did.
Conversely, a mediocre business in a powerful bull market can run up
for months. **You need both signals to make a confident decision.**

---

#### What "market volatility regime" means — for developers

Think of the stock market as a distributed system with two distinct
operating states:

| State | Analogy | Signal | What to do |
|---|---|---|---|
| **Low-vol / trending** | System idle, queue depth normal | [VIX](https://www.investopedia.com/terms/v/vix.asp) < 20, SPY above 200d MA | Act on your signals |
| **High-vol / choppy** | CPU spike, queue backing up | VIX > 25, SPY below 200d MA | Reduce size or wait |
| **Crash / panic** | Service outage | VIX > 35, SPY in freefall | Stop. Wait for recovery signal. |

The code cell below computes four checks against SPY (the S&P 500 ETF —
the closest thing to "the market"):

1. **Realized volatility** — SPY's annualised daily-return standard deviation
   over the last 21 trading days. > 25% = elevated; > 35% = dangerous.
   This is a real-time proxy for the VIX (fear index) using data you
   already have.

2. **Trend regime** — Is SPY above its 200-day moving average?
   Above = bull market; below = bear/correction. This one filter has
   historically kept you out of the worst crashes.

3. **Recent momentum** — Is SPY higher than it was 20 trading days ago?
   A negative number means you're fighting the tide.

4. **[Golden / Death Cross](https://www.investopedia.com/terms/g/goldencross.asp) on SPY** — SPY's 50-day MA vs 200-day MA.
   Golden Cross (50 > 200) = long-term uptrend; Death Cross = opposite.

Then it combines these into a three-tier **market verdict**:
`GREEN` / `YELLOW` / `RED`. The stock-level P7 score from §3 is only
actionable when the market verdict is GREEN or YELLOW.



In [8]:
# Cell 3.4 — Market timing check: is the current market safe to trade?
# Uses SPY (S&P 500 ETF) as the market proxy. No extra API calls if fmp_cached
# is warm — SPY data was already pulled in Phase 5.
import numpy as np
import pandas as pd
from openbb import obb
from pathlib import Path
import sys

# Re-use repo_root / sys.path from Cell 1.4 / 3.1 if already set.
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

GREEN  = "\033[92m"
YELLOW = "\033[93m"
RED    = "\033[91m"
BOLD   = "\033[1m"
RESET  = "\033[0m"
DIM    = "\033[2m"

# ── Pull SPY data (same window the Phase 5 benchmark uses) ─────────────────
import datetime
end   = datetime.date.today().isoformat()
start = (datetime.date.today() - datetime.timedelta(days=365)).isoformat()

spy_raw = obb.equity.price.historical(
    symbol="SPY", start_date=start, end_date=end,
    interval="1d", provider="fmp_cached"
)
spy = spy_raw.to_df() if hasattr(spy_raw, 'to_df') else pd.DataFrame(spy_raw.results)
spy.columns = [str(c).lower() for c in spy.columns]
if "date" in spy.columns and not isinstance(spy.index, pd.DatetimeIndex):
    spy = spy.set_index(pd.to_datetime(spy["date"])).sort_index()

close = spy["close"].dropna()
ret   = close.pct_change().dropna()

# ── Check 1: Realized volatility (21-day, annualised) ─────────────────────
# Think of this as your real-time fear gauge (VIX proxy).
# VIX < 20 → calm; 20-25 → caution; > 25 → elevated; > 35 → danger.
rv_21   = float(ret.rolling(21).std().iloc[-1] * np.sqrt(252))
rv_pct  = rv_21 * 100

if rv_pct < 15:
    rv_color, rv_label = GREEN,  f"Low      ({rv_pct:.1f}% ann.) — market is calm"
elif rv_pct < 20:
    rv_color, rv_label = GREEN,  f"Normal   ({rv_pct:.1f}% ann.) — typical operating range"
elif rv_pct < 25:
    rv_color, rv_label = YELLOW, f"Elevated ({rv_pct:.1f}% ann.) — reduce position size"
elif rv_pct < 35:
    rv_color, rv_label = YELLOW, f"High     ({rv_pct:.1f}% ann.) — caution, tight stops"
else:
    rv_color, rv_label = RED,    f"Danger   ({rv_pct:.1f}% ann.) — do NOT add new positions"

# ── Check 2: SPY above/below 200-day MA (bull vs bear regime) ─────────────
# The 200d MA is the single most-watched long-term trend line.
# Above it = participating in the bull; below it = swimming against the tide.
sma200 = float(close.rolling(200).mean().iloc[-1])
spy_last = float(close.iloc[-1])
above_200 = spy_last > sma200
sma200_gap = (spy_last / sma200 - 1) * 100

if above_200 and sma200_gap > 5:
    trend_color, trend_label = GREEN,  f"Bull regime  (SPY {sma200_gap:+.1f}% above 200d MA)"
elif above_200:
    trend_color, trend_label = YELLOW, f"Mild bull    (SPY {sma200_gap:+.1f}% above 200d MA — thin margin)"
elif sma200_gap > -5:
    trend_color, trend_label = YELLOW, f"Borderline   (SPY {sma200_gap:+.1f}% below 200d MA — watch closely)"
else:
    trend_color, trend_label = RED,    f"Bear regime  (SPY {sma200_gap:+.1f}% below 200d MA)"

# ── Check 3: Recent momentum — is SPY higher than 20 days ago? ────────────
# If the tide is going out, even good boats sink.
mom_20 = (close.iloc[-1] / close.iloc[-21] - 1) * 100 if len(close) >= 21 else float("nan")
if np.isnan(mom_20):
    mom_color, mom_label = YELLOW, "Insufficient data"
elif mom_20 > 2:
    mom_color, mom_label = GREEN,  f"Positive ({mom_20:+.1f}% over 20 days) — market climbing"
elif mom_20 > -1:
    mom_color, mom_label = YELLOW, f"Flat     ({mom_20:+.1f}% over 20 days) — no clear direction"
else:
    mom_color, mom_label = RED,    f"Negative ({mom_20:+.1f}% over 20 days) — market pulling back"

# ── Check 4: Golden/Death Cross on SPY ────────────────────────────────────
# Golden Cross: 50d MA crosses above 200d MA → sustained uptrend.
# Death Cross:  50d MA crosses below 200d MA → sustained downtrend.
sma50  = float(close.rolling(50).mean().iloc[-1])
golden = sma50 > sma200
cross_gap = (sma50 / sma200 - 1) * 100
if golden and cross_gap > 1:
    cross_color, cross_label = GREEN,  f"Golden Cross (50d {cross_gap:+.1f}% above 200d) — long-term uptrend"
elif golden:
    cross_color, cross_label = YELLOW, f"Borderline   (50d {cross_gap:+.1f}% above 200d) — recently crossed"
else:
    cross_color, cross_label = RED,    f"Death Cross  (50d {cross_gap:+.1f}% vs 200d) — long-term downtrend"

# ── Composite verdict ──────────────────────────────────────────────────────
greens  = sum(c == GREEN  for c in [rv_color, trend_color, mom_color, cross_color])
reds    = sum(c == RED    for c in [rv_color, trend_color, mom_color, cross_color])

if reds == 0 and greens >= 3:
    verdict_color = GREEN
    verdict = "GREEN  — Good conditions. Act on your P7 signal at normal size."
    action  = "Proceed using the position size from Phase 5 (recommended_size)."
elif reds <= 1 and greens >= 2:
    verdict_color = YELLOW
    verdict = "YELLOW — Proceed with reduced size and tighter stops."
    action  = "Halve Phase 5 recommended_size. Use hard stop at ATR × 1.5."
else:
    verdict_color = RED
    verdict = "RED    — Do not open new positions. Wait for conditions to improve."
    action  = "Stay in cash or paper-trade only. Re-run this cell daily."

# ── Print report ──────────────────────────────────────────────────────────
last_dt = close.index[-1]
last_date_str = last_dt.date() if hasattr(last_dt, 'date') and callable(last_dt.date) else last_dt
print(f"{BOLD}Market Timing Check — SPY as of {last_date_str}{RESET}\n")
print(f"  {'Realized vol (21d):':28} {rv_color}{rv_label}{RESET}")
print(f"  {'Trend regime (200d MA):':28} {trend_color}{trend_label}{RESET}")
print(f"  {'Recent momentum (20d):':28} {mom_color}{mom_label}{RESET}")
print(f"  {'Long-term cross (50/200):':28} {cross_color}{cross_label}{RESET}")
print()
print(f"  {BOLD}Verdict:{RESET} {verdict_color}{BOLD}{verdict}{RESET}")
print(f"  {DIM}Action:  {action}{RESET}")
print()

# ── Cross-reference with the Phase 7 stock score ─────────────────────────
# If p7 is available from Cell 3.2, show the combined signal.
try:
    p7 = results["p7"]
    p7_score = p7.composite_score
    p7_label = p7.action_label
    print(f"  {BOLD}Combined signal for {cfg.symbol}:{RESET}")
    print(f"    P7 company score : {p7_score:.2f}/5.0  →  {p7_label}")
    print(f"    Market verdict   : {verdict_color}{verdict.split('—')[0].strip()}{RESET}")

    # Combined decision logic
    if verdict_color == GREEN and p7_score >= 3.6:
        combined = f"{GREEN}{BOLD}TRADE  — both company health and market conditions are favourable.{RESET}"
    elif verdict_color == RED or p7_score < 2.8:
        combined = f"{RED}{BOLD}PASS   — one or both signals are negative. Wait.{RESET}"
    else:
        combined = f"{YELLOW}{BOLD}WATCH  — signals mixed. Paper-trade or use minimal size.{RESET}"
    print(f"    Combined verdict : {combined}")
except NameError:
    print(f"  {DIM}(Run Cell 3.1 first to get the P7 stock score for the combined verdict.){RESET}")


Market Timing Check — SPY as of 2026-06-26

  Realized vol (21d):          Normal   (16.5% ann.) — typical operating range
  Trend regime (200d MA):      Bull regime  (SPY +5.6% above 200d MA)
  Recent momentum (20d):       Negative (-3.4% over 20 days) — market pulling back
  Long-term cross (50/200):    Golden Cross (50d +6.3% above 200d) — long-term uptrend

  Verdict: YELLOW — Proceed with reduced size and tighter stops.
  Action:  Halve Phase 5 recommended_size. Use hard stop at ATR × 1.5.

  Combined signal for MSFT:
    P7 company score : 2.49/5.0  →  Avoid
    Market verdict   : YELLOW
    Combined verdict : PASS   — one or both signals are negative. Wait.


 3.4.1 How to read the market verdict — and what to do next

#### The gate rule

This check is a **hard gate** before §4. The logic mirrors a circuit-breaker
in software: if the system is under stress, you don't deploy new code —
you wait for stability.

```mermaid
flowchart TD
    F["§3 Fundamentals<br/>(P7 composite score)"] --> M["§3.4 Market check"]
    M --> GATE{"Market verdict?"}
    GATE -- "GREEN<br/>(all systems normal)" --> G["§4 Technical Setup<br/><b>at full recommended_size</b>"]
    GATE -- "YELLOW<br/>(1 amber flag)" --> Y["§4 Technical Setup<br/><b>halve size, ATR × 1.5 stop</b>"]
    GATE -- "RED<br/>(hi-vol / bear / death cross)" --> R["<b>STOP.</b><br/>Paper-trade only.<br/>Re-run §3.4 tomorrow."]

    classDef fundamentals fill:#7c3aed,stroke:#4c1d95,stroke-width:2px,color:#ffffff;
    classDef technical fill:#0891b2,stroke:#155e75,stroke-width:2px,color:#ffffff;
    classDef gate fill:#b45309,stroke:#7c2d12,stroke-width:2px,color:#ffffff;
    classDef green fill:#15803d,stroke:#14532d,stroke-width:2px,color:#ffffff;
    classDef yellow fill:#ca8a04,stroke:#713f12,stroke-width:2px,color:#ffffff;
    classDef red fill:#b91c1c,stroke:#7f1d1d,stroke-width:2px,color:#ffffff;

    class F fundamentals;
    class M technical;
    class GATE gate;
    class G green;
    class Y yellow;
    class R red;
```

#### Verdict reference card

| Verdict | Market conditions | What it means for you | Action |
|---|---|---|---|
| **GREEN** | Low vol + bull trend + positive momentum + golden cross | All systems normal. The wind is at your back. | Use Phase 5 `recommended_size` as-is |
| **YELLOW** | 1 amber flag or borderline readings | The market is uncertain. Your analysis may still be right but timing is harder. | Halve size. Set stop at ATR × 1.5. |
| **RED** | High vol OR bear regime OR death cross | The market itself is the risk. Individual stock signals are unreliable. | **Do not open new positions.** Paper-trade only. |

#### What "reduced size and tighter stops" actually means

If you're new to trading, the YELLOW action line can feel cryptic.Two concepts are at work:


- **Reduced size** — buy fewer shares than the plan recommends.
  If Phase 5 says `recommended_size = 100 shares`, you buy 50.
  This caps your total dollar exposure when the market is uncertain.
  *Developer analogy:* rolling out a feature to a 5% canary groupinstead of 100% of users — same code, smaller blast radius.

- **Tighter stops** — move your stop-loss price closer to your entry.
  Normally techtrade sets the stop at `entry − [ATR](https://www.investopedia.com/terms/a/atr.asp) × 2` (two
  Average True Ranges below entry). "Tighter" means `ATR × 1.5` —
  a smaller cushion. If the stock moves against you even slightly,
  you exit sooner, losing less per share.
  *Developer analogy:* lowering your error-rate threshold from 5%
  to 2% before the circuit breaker trips — you tolerate less noise
  before pulling the plug.

Together: you're risking **less money** (fewer shares) and giving the
trade **less room to go wrong** (closer stop). The total capital at
risk drops to roughly ¼ of a GREEN-verdict trade.

#### Why technical analysis is useless in a RED market

When the market crashes (or panics), correlations go to 1.0 — meaning every
stock moves down together regardless of its fundamentals. In software terms:
you're debugging an individual microservice, but the entire cluster is on fire.
Fix the cluster first.

Running `obb.techtrade.signals` / `plan` / `scan` during a RED market will
produce signals, but those signals have almost no edge — they'll be drowned by
the macro tide. The validation engine (`obb.techtrade.validate`) exists
precisely to measure this, but it runs on historical data, not the current
regime. **The market timing check is your current-regime filter.**

#### When to re-check

Re-run the cell above whenever:
- It has been > 3 trading days since you last ran it
- You see an unusual market event (big Fed announcement, earnings shock, geopolitical news)
- You're about to size into a new position from §4

> **Developer analogy:** Think of this cell as a `health_check()` call you
> make before every `deploy()`. You wouldn't skip it just because yesterday's
> health check passed.



## 4. Technical Setup — the techtrade engine

> **Pre-condition:** §3.4 market verdict must be **GREEN or YELLOW** before
> running any cell in this section. If it returned RED, stop here and
> come back when market conditions improve. There is no signal strong enough
> to overcome a RED market.

### What this section is teaching

Section 3 answered "should I be trading at all right now?" This section answers **"which stock, in which direction, at what size, and where's the exit?"** — the four questions every trader has to answer before clicking Buy or Sell.

techtrade is a **[technical analysis](https://www.investopedia.com/terms/t/technicalanalysis.asp) engine** — it makes decisions from price and volume history alone, not from company financials. That's a deliberate scope: [fundamental analysis](https://www.investopedia.com/terms/f/fundamentalanalysis.asp) (revenue, earnings, moats) tells you *what* is worth owning; technical analysis tells you *when* to enter and exit. Section 5 covers fundamentals; this section is timing.

Its job is to turn [OHLCV bars](https://www.investopedia.com/terms/o/ohlcchart.asp) (**O**pen / **H**igh / **L**ow / **C**lose / **V**olume per fixed time window — a [candlestick](https://www.investopedia.com/terms/c/candlestick.asp) with a volume bar underneath) into an actionable, **auditable** trade plan: entry [levels](https://www.investopedia.com/terms/s/support.asp), [stops](https://www.investopedia.com/terms/s/stop-lossorder.asp), position sizing, and an Excel-ready summary. Every score it produces carries the full vote attribution so Alex can always answer *"why long?"* — this is what auditability means in trading, and it's the difference between a disciplined system and a gambling hunch.

### The four-layer architecture (PRD §12)

```mermaid
flowchart LR
    IN["<b>OHLCV bars</b><br/>(open/high/low/close/vol<br/>per fixed time window)"]
    IND["<b>Indicators</b><br/>engine/indicators.py<br/><i>facts from price + volume</i>"]
    CONF["<b>Confluence</b><br/>engine/confluence.py<br/><i>weighted vote → score</i>"]
    RULES["<b>Rules + Orders</b><br/>engine/rules.py + plan.py<br/><i>entry / stop / target / time-exit</i>"]
    BROKER["<b>Paper broker</b><br/>execution.py<br/><i>next-bar fills, slippage, commission</i>"]
    OUT["<b>long / short / flat</b><br/>+ entry, stop, target"]

    IN --> IND --> CONF --> RULES --> BROKER --> OUT

    classDef input fill:#2563eb,stroke:#1e3a8a,stroke-width:2px,color:#ffffff;
    classDef facts fill:#7c3aed,stroke:#4c1d95,stroke-width:2px,color:#ffffff;
    classDef opinion fill:#0891b2,stroke:#155e75,stroke-width:2px,color:#ffffff;
    classDef action fill:#15803d,stroke:#14532d,stroke-width:2px,color:#ffffff;
    classDef simulate fill:#b45309,stroke:#7c2d12,stroke-width:2px,color:#ffffff;
    classDef output fill:#b91c1c,stroke:#7f1d1d,stroke-width:2px,color:#ffffff;

    class IN input;
    class IND facts;
    class CONF opinion;
    class RULES action;
    class BROKER simulate;
    class OUT output;
```

1. **Indicators** (`engine/indicators.py`) — computes ~10 classical [technical indicators](https://www.investopedia.com/terms/t/technicalindicator.asp) grouped into four families (trend / momentum / volatility / volume) using the vendored `pandas-ta-classic` library. Output: an `IndicatorPanel` — a plain Python dict of latest indicator values per symbol.
2. **Confluence** (`engine/confluence.py`) — turns those raw indicator values into individual **votes** (each indicator votes long / short / flat), then combines them into one composite `score ∈ [-1, +1]` using the locked family weights (`trend 0.40 / momentum 0.25 / volatility 0.20 / volume 0.15`). Three presets ship: `trend_follow` (default), `mean_revert`, `breakout` — same voters, different weight bias.
3. **Rules + Orders** (`engine/rules.py` + `engine/plan.py`) — takes the signal + an `EntryExitRule` and generates concrete orders: entry price, [stop-loss](https://www.investopedia.com/terms/s/stop-lossorder.asp), [take-profit](https://www.investopedia.com/terms/t/take-profitorder.asp), and a [time exit](https://www.investopedia.com/terms/t/time-stop.asp), all sized by [risk-per-trade](https://www.investopedia.com/terms/r/risk-rewardratio.asp) (default 1% of notional).
4. **Paper broker** (`execution.py`) — simulates fills at next-bar-open with realistic [slippage](https://www.investopedia.com/terms/s/slippage.asp) + [commission](https://www.investopedia.com/terms/c/commission.asp). **No [look-ahead bias](https://www.investopedia.com/terms/l/lookaheadbias.asp)** — bar-*t* signals only fill at *t+1*, which is the single most important discipline in [backtesting](https://www.investopedia.com/terms/b/backtesting.asp) (using future information to make a past decision is the #1 source of "great backtest, terrible live results").

### The indicator catalog — what each one measures and votes on

Every indicator falls into one of four families. The engine computes ~10 by default; **~8 of them actually vote** on the composite score, the other 2 act as gates or stop-sizing inputs. Beginners often ask "why not just use RSI?" — the answer is that no single indicator is right in every market regime, and the four families each measure something different, so when they agree the signal is much stronger than any one alone (that's what *confluence* means — see cell 22 below).

#### Trend family — "is price persistently moving in one direction?"

| Indicator | Default period | What it measures | How it votes |
|---|---|---|---|
| [**MACD** (histogram)](https://www.investopedia.com/terms/m/macd.asp) | 12/26/9 | Difference between fast and slow [exponential moving averages](https://www.investopedia.com/terms/e/ema.asp), minus its own signal line. Positive = fast EMA above slow EMA (bullish); rising = momentum accelerating | `+1` if histogram > 0, `-1` if < 0 — gated by ADX below |
| [**ADX** (Average Directional Index)](https://www.investopedia.com/terms/a/adx.asp) | 14 | *Strength* of the trend, direction-agnostic. > 25 = strong trend; < 20 = choppy / [range-bound](https://www.investopedia.com/terms/r/rangeboundtrading.asp) | **Not a voter — a gate.** If ADX ≤ 20 the trend votes get scaled down (`clip(adx/20, 0, 1)`) because a weak trend shouldn't count as a full-strength vote |
| [**EMA cross**](https://www.investopedia.com/terms/g/goldencross.asp) | 20/50 | Sign of fast EMA − slow EMA. When fast crosses above slow ([golden cross](https://www.investopedia.com/terms/g/goldencross.asp)), classic long-term bullish signal | `+1` if fast > slow, `-1` if fast < slow |

**Why trend gets the biggest family weight (0.40):** most systematic edges depend on price *continuing* in a direction. [Momentum strategies](https://www.investopedia.com/terms/m/momentum_investing.asp) empirically beat [mean-reversion](https://www.investopedia.com/terms/m/meanreversion.asp) in equity indices over multi-year horizons — that's why the default preset (`trend_follow`) leans this way.

#### Momentum family — "is the rate of change accelerating?"

| Indicator | Default period | What it measures | How it votes |
|---|---|---|---|
| [**RSI** (Relative Strength Index)](https://www.investopedia.com/terms/r/rsi.asp) | 14 | Ratio of average up-move to average down-move over N periods, on 0-100 scale. > 70 = [overbought](https://www.investopedia.com/terms/o/overbought.asp), < 30 = [oversold](https://www.investopedia.com/terms/o/oversold.asp) | `+1` if RSI > 50 with room to run (~40-60 pullback zone), `-1` symmetrically below 50; extreme values are dampened |
| [**Stochastic**](https://www.investopedia.com/terms/s/stochasticoscillator.asp) | 14/3/3 (K/D/smooth) | Where today's close sits within the recent high-low range, on 0-100 scale. %K crossing above %D from below 20 = classic bullish turn from oversold | `+1` if %K > %D, `-1` if %K < %D |

**Why momentum gets 0.25 (second-largest):** momentum is the **confirming** vote. When trend says "yes" AND momentum says "yes," the signal is much less likely to be a false start. When they disagree — trend up, momentum stalling — that's a warning that the trend is losing energy, and the composite score drops accordingly.

#### Volatility family — "is the market moving enough to trade, but not so much it stops us out?"

| Indicator | Default period | What it measures | How it votes |
|---|---|---|---|
| [**Bollinger Band %B**](https://www.investopedia.com/terms/b/bollingerbands.asp) | 20, 2σ | Where price sits within the [Bollinger Bands](https://www.investopedia.com/terms/b/bollingerbands.asp) (moving average ± 2 standard deviations). 0 = at lower band, 1 = at upper band | `+1` if %B > 0.5 (upper half of the range — bullish continuation), `-1` if < 0.5 |
| [**ATR** (Average True Range)](https://www.investopedia.com/terms/a/atr.asp) | 14 | Typical size of one day's price move in dollars. Used by the [rules layer](https://www.investopedia.com/terms/s/stop-lossorder.asp) to place stops (e.g. 2 × ATR below entry) | **Not a voter — a stop-sizing input.** A stock with $5 ATR needs a wider stop than one with $0.50 ATR; ATR normalizes that |
| [**Keltner Channels**](https://www.investopedia.com/terms/k/keltnerchannel.asp) | 20, scalar 2 | Similar to Bollinger Bands but built from EMA ± ATR multiples (less sensitive to volatility spikes) | Currently informational; used by the `mean_revert` preset for stretch detection |

**Why volatility gets 0.20 (third):** volatility is context — the same trend signal in a calm market vs. a wild one demands different position sizes. It also gates the whole system: if [realized volatility](https://www.investopedia.com/terms/r/realizedvolatility.asp) is too high, stops get blown out on normal noise; too low, and the market isn't moving enough to profit.

#### Volume family — "are participants actually showing up?"

| Indicator | Default period | What it measures | How it votes |
|---|---|---|---|
| [**OBV** (On-Balance Volume) — slope](https://www.investopedia.com/terms/o/onbalancevolume.asp) | slope over 10 bars | Running total that adds volume on up-days, subtracts on down-days. Rising OBV = [accumulation](https://www.investopedia.com/terms/a/accumulation-distribution.asp) (smart money buying); falling OBV = distribution. Slope makes it a directional measure | `+1` if slope > 0, `-1` if < 0 |
| [**CMF** (Chaikin Money Flow)](https://www.investopedia.com/terms/c/chaikinmoneyflow.asp) | 20 | Volume-weighted measure of buying vs. selling pressure. > 0 = money flowing in, < 0 = money flowing out | `+1` if CMF > 0, `-1` if < 0 |

**Why volume gets the smallest weight (0.15):** volume is a *confirmer*, not a *leader*. Volume can dry up in a legitimate uptrend (low-volume drift higher), and volume can spike on false breakouts. A rising OBV during a rising price is reassuring; a falling OBV during a rising price ([bearish divergence](https://www.investopedia.com/terms/d/divergence.asp)) is a warning — but neither alone is enough to trade on. Volume gets a small vote so it can *tip the scale* on borderline signals but not drive them.

### How confluence produces the final score

Once every indicator has cast its `+1 / -1 / 0` vote, the engine computes:

```
score = Σ (vote_i × family_weight_i × per_family_normalization)
      → clipped to [-1, +1]
```

The full voting flow — from the eight voters through the family weights to
the final direction band:

```mermaid
flowchart LR
    subgraph TR["Trend family — weight 0.40"]
        direction TB
        MACD["MACD histogram<br/>(12/26/9)"]
        EMA["EMA cross<br/>(20/50)"]
        ADX["ADX(14)<br/><i>gate: scales trend votes<br/>if ADX ≤ 20</i>"]
    end
    subgraph MO["Momentum — weight 0.25"]
        direction TB
        RSI["RSI(14)"]
        STOCH["Stochastic<br/>(14/3/3)"]
    end
    subgraph VOL["Volatility — weight 0.20"]
        direction TB
        BBP["Bollinger %B<br/>(20, 2σ)"]
        KELT["Keltner(20,2)<br/><i>informational</i>"]
        ATR["ATR(14)<br/><i>stop-sizing input</i>"]
    end
    subgraph VU["Volume — weight 0.15"]
        direction TB
        OBV["OBV slope<br/>(10 bars)"]
        CMF["CMF(20)"]
    end

    TR --> SCORE["<b>Composite score</b><br/>Σ (vote × weight)<br/>clipped to [-1, +1]"]
    MO --> SCORE
    VOL --> SCORE
    VU --> SCORE

    SCORE --> BAND{"Direction<br/>threshold"}
    BAND -- "score ≥ +0.6" --> LONG["<b>LONG</b>"]
    BAND -- "-0.6 &lt; score &lt; +0.6" --> FLAT["<b>FLAT</b><br/>(stand aside)"]
    BAND -- "score ≤ -0.6" --> SHORT["<b>SHORT</b>"]

    classDef trend fill:#7c3aed,stroke:#4c1d95,stroke-width:2px,color:#ffffff;
    classDef momentum fill:#0891b2,stroke:#155e75,stroke-width:2px,color:#ffffff;
    classDef volatility fill:#b45309,stroke:#7c2d12,stroke-width:2px,color:#ffffff;
    classDef volume fill:#2563eb,stroke:#1e3a8a,stroke-width:2px,color:#ffffff;
    classDef score fill:#4b5563,stroke:#1f2937,stroke-width:2px,color:#ffffff;
    classDef long fill:#15803d,stroke:#14532d,stroke-width:2px,color:#ffffff;
    classDef flat fill:#6b7280,stroke:#374151,stroke-width:2px,color:#ffffff;
    classDef short fill:#b91c1c,stroke:#7f1d1d,stroke-width:2px,color:#ffffff;

    class MACD,EMA,ADX trend;
    class RSI,STOCH momentum;
    class BBP,KELT,ATR volatility;
    class OBV,CMF volume;
    class SCORE,BAND score;
    class LONG long;
    class FLAT flat;
    class SHORT short;
```

**How to read this:** ADX is a *gate* on the trend family (a weak trend
gets scaled down), ATR feeds stop-sizing (not the vote), Keltner is
currently informational. The other eight — MACD, EMA-cross, RSI,
Stochastic, %B, OBV slope, CMF — cast the actual votes.

Then it maps to a direction:

| Score band | Direction | Meaning |
|---|---|---|
| `score ≥ +0.6` | **long** | Strong agreement across families that the odds favor a rise |
| `+0.6 > score > -0.6` | **flat** | Not enough agreement; stand aside (the null hypothesis is "do nothing") |
| `score ≤ -0.6` | **short** | Strong agreement that odds favor a fall — position for a decline (or, if [long-only](https://www.investopedia.com/terms/l/long.asp), simply exit) |

The default `±0.6` threshold means **at least 3-4 indicators across ≥3 families have to agree** before a trade fires. This is intentionally conservative — cutting most of the small-score noise trades that historically ate returns in the [live vs backtest gap](https://www.investopedia.com/terms/b/backtesting.asp).

### The three presets — same voters, different bias

The same 8 voters get re-weighted per market regime:

| Preset | When to use | What shifts | Rationale |
|---|---|---|---|
| **`trend_follow`** (default) | GREEN market, indexes above their 200-day [SMA](https://www.investopedia.com/terms/s/sma.asp) | Baseline weights (`0.40/0.25/0.20/0.15`) | Standard "buy strength, sell weakness" — the mode that works in trending regimes |
| **`mean_revert`** | Range-bound / choppy market (YELLOW-ish) | Bollinger %B and Stochastic get more weight; MACD gets less | In a [range](https://www.investopedia.com/terms/r/rangeboundtrading.asp), the edges of the Bollinger band are more informative than a moving-average cross that keeps whipsawing |
| **`breakout`** | Post-consolidation, expanding [range expansion](https://www.investopedia.com/terms/r/range.asp) | Volume and volatility get more weight; RSI gets less | When price breaks out of a squeeze, [volume confirmation](https://www.investopedia.com/terms/b/breakout.asp) matters more than whether RSI is at 70 |

`obb.techtrade.tune` (issue #83, shipped) can **refit these weights per market segment** with a robustness gate — so you don't keep weights that only worked on past data. See §4.6 for the tuning walkthrough.

### What "auditable" means — every score explains itself

Every `signals` / `plan` call returns not just a score but **the individual votes** that produced it. That means Alex can always answer *"why did this fire?"* by looking at:

```
symbol   score   direction   trend_votes   momentum_votes   volatility_votes   volume_votes
MSFT     +0.72   long        +1,+1,+1     +1,+1           +1                 +1,+1
```

*"MSFT is long because all three trend indicators agree, both momentum indicators agree, Bollinger %B is in the upper half, and both volume indicators show accumulation."* This is what the composite `score ∈ [-1, +1]` means in practice — it's a *summary* of an underlying vote panel, not an opaque black-box number. Cell 4.1 below shows the raw output.

### One more discipline — the auto-tuner won't let you overfit

The [family weights](https://www.investopedia.com/terms/o/overfitting.asp) shown above (0.40/0.25/0.20/0.15) are the "tested at scale" defaults. `obb.techtrade.tune` re-fits them per segment (e.g. tech stocks might want higher momentum weight; utilities might want more volatility) — but only if the retuned weights pass a **[walk-forward validation](https://www.investopedia.com/articles/trading/11/backtesting-walkforward-important-correlation.asp)** gate: the weights must produce positive risk-adjusted returns on **out-of-sample data**, not just on the training window. If they don't, the tuner rejects them and keeps the defaults. This is the mechanical difference between "we found a pattern that worked in the past" (overfitting) and "we found a pattern that continues to work" (real edge).


> **What "confluence" means — and why these weights**
>
> *Confluence* is just "several independent indicators *agreeing* on
> direction." techtrade's `score ∈ [-1, +1]` is a weighted vote:
>
> | Family | Weight | What it measures | Why this weight |
> |---|---|---|---|
> | Trend | 0.40 | Is price persistently above its [moving averages](https://www.investopedia.com/terms/m/movingaverage.asp)? | Most systematic edges depend on price *continuing* in a direction, so trend gets the biggest vote |
> | Momentum | 0.25 | Is the rate-of-change accelerating? | A confirming signal — trend "yes" + momentum "yes" reduces false starts |
> | Volatility | 0.20 | Is realized [ATR](https://www.investopedia.com/terms/a/atr.asp) in a tradeable range? | Stops out faster when vol exploding; gives no room in dead markets |
> | Volume | 0.15 | Are participants showing up? | A confirmer, not a leader — gets the smallest vote |
>
> *Software analogy:* think test pyramid voting — a feature ships when
> unit + integration + e2e all green, not when one alone passes. These
> weights are the "tested at scale" defaults; **`obb.techtrade.tune`
> (issue #83, now shipped) re-fits them per segment** with a robustness
> gate, so you don't keep weights that only worked on past data.


In [9]:
# Cell 4.1 — single-symbol confluence signal for MSFT.
# This call goes to fmp_cached for ~250 bars of daily OHLCV.
result = obb.techtrade.signals(symbols=["MSFT"], preset="trend_follow")
# `results` is a list[MoverSignal] — one per ranked signal.
signals = result.results
for s in signals:
    print(f"{s.symbol:6} score={s.score:+.3f}  direction={s.direction:<6}  votes={len(s.votes)}")

MSFT   score=-0.407  direction=short   votes=7


**What you should see:** one row with MSFT, a directional `score`
in `[-1, +1]`, and a `direction` of `long` / `short` / `flat`.
`votes` is a `list[IndicatorVote]` carrying the per-indicator
contribution to the score — that's what makes the call auditable.

### 4.2 The actionable call — `plan`

`plan` runs the same signal chain but assembles a complete
`TradePlan` per symbol: levels, sizing, broker-ready orders, and an
inline `Recommendation` (the human-facing call). It's the bridge
from "what's the signal" to "what do I actually do."


> **`direction` decoded — long / short / flat**
>
> - **long** — bet price will *rise*; buy shares now, sell later for (hopefully) more.
> - **short** — bet price will *fall*; borrow shares from your broker, sell now,
>   buy them back later (hopefully cheaper) to return. See
>   [Investopedia on shorting](https://www.investopedia.com/terms/s/shortselling.asp).
>   *Software analogy:* shorting is the financial version of compiling a
>   negative test — you profit when the system fails (price drops).
> - **flat** — no position; the signal isn't strong enough either way. The
>   discipline most traders are missing: **flat is a valid call**, not a
>   coward's call.


In [10]:
# Cell 4.2 — full trade plan for MSFT at 1% risk per trade.
result = obb.techtrade.plan(
    symbols=["MSFT"],
    preset="trend_follow",
    risk=0.01,
)
plans = result.results
for plan in plans:
    rec = plan.recommendation
    print(f"--- {plan.symbol} ({plan.segment}) as of {plan.as_of} ---")
    print(f"  Signal score:  {plan.signal.score:+.3f}  ->  {rec.action}  ({rec.conviction} conviction)")
    print(f"  Entry:         ${rec.entry_price}")
    print(f"  Stop:          ${rec.stop_price}      ({rec.stop_distance_pct*100:.2f}% away)")
    print(f"  Target:        ${rec.target_price}      ({rec.target_distance_pct*100:.2f}% away)")
    print(f"  R:R:           {rec.risk_reward:.2f}")
    print(f"  Position:      {rec.position_size} shares")
    print(f"  ATR(14):       {rec.atr:.2f}")
    print(f"  Time stop:     {rec.time_stop_bars} bars")
    print(f"  Caveats:       {rec.caveats}")
    print(f"  Orders ({len(plan.orders)}):")
    for o in plan.orders:
        print(f"    {o.intent:<12}  {o.side:<5}  qty={o.quantity}  type={o.order_type}")

--- MSFT (custom) as of 2026-06-26 ---
  Signal score:  -0.407  ->  SELL_SHORT  (Medium conviction)
  Entry:         $372.97
  Stop:          $399.3875105050188260      (7.08% away)
  Target:        $320.13497898996234800      (14.17% away)
  R:R:           2.00
  Position:      37 shares
  ATR(14):       13.21
  Time stop:     20 bars
  Caveats:       No paper fill - levels are planned, not realized. Volume diverging from price. Momentum votes split. Borderline conviction.
  Orders (5):
    entry         sell_short  qty=37  type=market
    exit_stop     buy_to_cover  qty=37  type=stop
    exit_target   buy_to_cover  qty=37  type=limit
    exit_time     buy_to_cover  qty=37  type=market
    exit_signal   buy_to_cover  qty=37  type=market


**What this is teaching Alex:** every order has an `intent` tag — [stop-loss](https://www.investopedia.com/terms/s/stop-lossorder.asp) exits live alongside the target ([R:R](https://www.investopedia.com/terms/r/riskrewardratio.asp) ratio shown in §4.2 above);
(`entry` / `exit_stop` / `exit_target` / `exit_time` / `exit_signal`).
A real broker integration knows which leg is which. The
paper broker (§4.5) respects this tagging too.

### 4.3 The cross-sector view — `scan`

`scan` runs `plan` across all 11 GICS sectors and ranks the
top setups by conviction. This is the daily "what does the universe
look like today" call.


> **What "1 % risk per trade" actually buys (worked example)**
>
> The `risk=0.01` parameter is **not** "spend 1 % of capital on this trade" —
> it's "lose 1 % of capital if the stop is hit." The position size is derived,
> not chosen. With a $10,000 account at 1 % risk on a $300 stock with ATR
> of $13:
>
> ```
> capital_at_risk = $10,000 × 1%   = $100
> stop_distance   = ATR × 2        = $26
> position_size   = $100 / $26     ≈ 3 shares   (rounds down)
> capital_used    = 3 × $300       = $900       (~ 9 % of account)
> ```
>
> See [position sizing on Investopedia](https://www.investopedia.com/articles/trading/09/determine-position-size.asp).
> The cap on dollar *exposure* is implicit; the explicit cap is dollar *loss*.
>
> **Time stop** — `time_stop_bars` is a third exit alongside stop and target:
> if neither has hit after N bars, the thesis didn't play out and the trade
> exits anyway. The number is wrong even if the price hasn't proved it yet.


In [11]:
# Cell 4.3 — segment scan across all 11 GICS sectors.
# WALL-CLOCK: ~30s on a warm cache; up to a few minutes cold.
result = obb.techtrade.scan(preset="trend_follow", risk=0.01)
plans = result.results
print(f"Scan returned {len(plans)} actionable plans.\n")
# Top-5 by absolute score.
top = sorted(plans, key=lambda p: -abs(p.signal.score))[:5]
for p in top:
    rec = p.recommendation
    print(
        f"  {p.symbol:6} {p.segment:<24} "
        f"score={p.signal.score:+.3f}  "
        f"{rec.action:<10}  R:R={rec.risk_reward:.1f}  "
        f"conv={rec.conviction}"
    )

Failed to fetch data from FMP for EtfHoldings: Unauthorized FMP request -> 402 -> Restricted Endpoint: This endpoint is not available under your current subscription please visit our subscription page to upgrade your plan at https://financialmodelingprep.com/
Failed to fetch data from FMP for EtfHoldings: Unauthorized FMP request -> 402 -> Restricted Endpoint: This endpoint is not available under your current subscription please visit our subscription page to upgrade your plan at https://financialmodelingprep.com/
Failed to fetch data from FMP for EtfHoldings: Unauthorized FMP request -> 402 -> Restricted Endpoint: This endpoint is not available under your current subscription please visit our subscription page to upgrade your plan at https://financialmodelingprep.com/
Failed to fetch data from FMP for EtfHoldings: Unauthorized FMP request -> 402 -> Restricted Endpoint: This endpoint is not available under your current subscription please visit our subscription page to upgrade your pla

Scan returned 66 actionable plans.

  EURKR  Communication Services   score=+0.664  BUY         R:R=2.0  conv=Medium
  EURKR  Consumer Discretionary   score=+0.664  BUY         R:R=2.0  conv=Medium
  EURKR  Consumer Staples         score=+0.664  BUY         R:R=2.0  conv=Medium
  EURKR  Energy                   score=+0.664  BUY         R:R=2.0  conv=Medium
  EURKR  Financials               score=+0.664  BUY         R:R=2.0  conv=Medium


### 4.4 Excel export — what Alex actually prints

`export` writes a 6-sheet Excel workbook (Recommendations / Levels /
Reasoning / Orders / Fills / Summary) with conditional formatting
(color-coded actions, R:R color scale, conviction). Default path is
`Analysis/exports/techtrade_<date>.xlsx`. Every workbook carries the
"research/paper output — not investment advice" disclaimer on the
Recommendations sheet.


In [12]:
# Cell 4.4 — export the scan to a multi-sheet Excel workbook.
from pathlib import Path

result = obb.techtrade.export(plans=plans)
xlsx_path = Path(result.results)
print(f"Wrote workbook: {xlsx_path}")
print(f"  Size: {xlsx_path.stat().st_size / 1024:.1f} KB")

# Peek at the sheet names so Alex knows what's in the file before opening it.
import openpyxl
wb = openpyxl.load_workbook(xlsx_path, read_only=True)
print(f"  Sheets ({len(wb.sheetnames)}): {wb.sheetnames}")
wb.close()

Wrote workbook: H:\masterswork\git\OpenBBTechnical\Analysis\exports\techtrade_2026-06-26.xlsx
  Size: 30.0 KB
  Sheets (6): ['Recommendations', 'Levels', 'Reasoning', 'Orders', 'Fills', 'Summary']


> Open the workbook in Excel / LibreOffice / Google Sheets to see
> the conditional formatting. The `Recommendations` sheet carries
> the per-row action/conviction; `Reasoning` holds the audit trail
> (which indicator votes drove the call); `Summary` is the
> end-of-scan summary.

### 4.5 Paper-broker simulation — `simulate`

For one plan, you can paper-fill its orders against a forward OHLCV
window. The fills include slippage + commission. **No look-ahead** —
a bar-*t* signal only fills at *t+1*.

(Skipping a live demo cell here because `simulate` requires a
forward bar window the cache may not have; see
`tests/integration/test_scan_integration.py` for a fixture-backed
example.)


## 5. The Robustness Gate — `obb.techtrade.validate` (issue #82)

This is the **anti-hunch** gate. Even a beautifully-formatted plan
from §4 might be fitting noise. `validate` delegates to the
in-tree `openbb-backtest` extension to ask the blunt question:

> **Would this rule have survived out-of-sample over the last 5 years?**

It re-runs the techtrade confluence strategy over [walk-forward (WFO)](https://www.investopedia.com/terms/b/backtesting.asp) or [combinatorial purged cross-validation (CPCV)](https://www.investopedia.com/terms/b/backtesting.asp) folds ([López de Prado paper](https://arxiv.org/abs/1603.02804)),
computes:

- **PBO — Probability of [Backtest Overfitting](https://www.investopedia.com/terms/o/overfitting.asp)** (López de Prado; [original paper](https://arxiv.org/abs/1304.4992)).
  Low is good (< 0.2 = robust; ≥ 0.5 = overfit).
- **DSR — Deflated [Sharpe Ratio](https://www.investopedia.com/terms/s/sharperatio.asp)** (Bailey / López de Prado; [original paper](https://www.davidhbailey.com/dhbpapers/deflated-sharpe.pdf)). High
  is good (> 0.95 = robust; < 0.5 = overfit).
- **OOS Sharpe** — aggregated out-of-sample Sharpe ratio.

...then returns a single `verdict ∈ {robust, fragile, overfit}`
with the precedence **overfit > robust > fragile**.

> **What "fragile" actually means**: not enough evidence to call
> the edge real, but no evidence it's fake either. Treat fragile
> setups as "size smaller and watch behavior." `overfit` is a
> hard no.

### 5.1 Validate the MSFT plan

**Wall-clock warning:** 2-10 minutes on a warm cache. This cell
calls real WFO folds; each fold re-runs the strategy over a
multi-year OHLCV window. If you're just reading, skip it.


> **Plain-English PBO & DSR — for developers**
>
> Both numbers exist because backtests look better in-sample than out-of-sample
> for *systematic* reasons, not just unlucky ones.
>
> - **PBO ≈ "did I just get lucky on this fold split?"** It re-shuffles your
>   train/test folds many times and counts what fraction of shuffles would have
>   produced an in-sample winner this strong. High PBO = the in-sample-best
>   strategy is *expected* to underperform OOS. *Software analogy:* unit tests
>   that pass only on your dev box — would they pass on a fresh CI runner?
> - **DSR ≈ "Sharpe ratio after a haircut for how many strategies I tried."**
>   The vanilla [Sharpe ratio](https://www.investopedia.com/terms/s/sharperatio.asp)
>   is inflated by multiple-testing: if you tried 100 strategies, the best one
>   *looks* great even when none has true edge. DSR penalises for the search
>   you did. *Software analogy:* p-value Bonferroni correction.


In [13]:
# Cell 5.1 — validate the first MSFT plan from §4.2.
# Choose the MSFT plan from §4.2 (re-run plan if `plans` is empty).
plan_to_validate = next((p for p in plans if p.symbol == "MSFT"), None)
if plan_to_validate is None:
    # Fall back: re-build a single-symbol MSFT plan if scan didn't include it.
    plan_to_validate = obb.techtrade.plan(symbols=["MSFT"]).results[0]

from openbb_techtrade.validation.backtest_bridge import validate_plan

# validate_plan is async because the WFO loop can take minutes. The Jupyter
# kernel already runs inside an asyncio event loop, so we `await` the coroutine
# directly (top-level await) instead of asyncio.run(), which would raise
# "asyncio.run() cannot be called from a running event loop".
updated_plan, report = await validate_plan(
    plan_to_validate, method="wfo", horizon_years=5
)
print(f"Verdict:               {report.verdict}")
print(f"Method:                {report.method}")
print(f"PBO (lower=better):    {report.pbo:.3f}")
print(f"Deflated Sharpe:       {report.deflated_sharpe:.3f}")
print(f"OOS Sharpe:            {report.oos_metrics.sharpe:.3f}")
print(f"Min-backtest-length:   {report.min_backtest_length_years:.2f} years")
print(f"Folds: {len(report.folds)}")
print(f"\nVerdict thresholds in effect:")
for k, v in report.thresholds.items():
    print(f"  {k:<14} {v}")

KeyError: "no strategy registered as 'techtrade_confluence'"

**How to read this output:**

- **`verdict: robust`** — the strategy survived OOS testing. The
  gate would approve persisting tuned parameters for it (see §6).
- **`verdict: fragile`** — couldn't reject "this is noise" with
  statistical confidence. Don't bet the farm.
- **`verdict: overfit`** — actively bad. The rule looks great
  in-sample because it memorized the past; OOS performance is poor.
  Walk away.

### 5.2 What `validate` does NOT do

- It does **not** tell you the strategy will work tomorrow.
  "Robust over 5 years" is a *necessary*, not *sufficient*,
  condition for shipping a rule.
- It does **not** tune the rule. The tuner is `obb.techtrade.tune`
  (issue #83, roadmap — see §6).
- It does **not** test on your live order book or your slippage
  profile. Forward-test on paper first (`simulate`) before scaling.


## 6. Roadmap — what's shipped and what's next

techtrade ships in phases (PRD §18). Refreshed as of **2026-06-27**:

| Phase | Status | Issue | Surface |
|---|---|---|---|
| P0 Foundations | ✅ Shipped | #65 | Extension skeleton, models, [GICS](https://www.investopedia.com/terms/g/gics.asp) ([owned by MSCI](https://www.msci.com/index/methodology/latest/GICS)) map |
| P1 Segment screener | ✅ Shipped | #69 | `obb.techtrade.segments`, `movers` |
| P2 Indicator engine | ✅ Shipped | #72/#73 | `engine.indicators`, `pandas-ta-classic` adapter |
| P3 Confluence engine | ✅ Shipped | #74/#75 | `obb.techtrade.signals`, 3 presets |
| P4 Rules + orders + fills | ✅ Shipped | #77/#78 | `obb.techtrade.plan`, `orders`, `simulate` |
| P5 Recommendation + Excel | ✅ Shipped | #80/#81 | `obb.techtrade.export`, 6-sheet workbook |
| P6 Validation bridge | ✅ Shipped 2026-06-21 | #82 | `obb.techtrade.validate` (covered in §5) |
| **P7 Tuning** | ✅ Shipped 2026-06-25 | **#83** | `obb.techtrade.tune` — all 7 tasks landed; gated WFO/CPCV + persistence |
| P8 Agent / MCP | 📋 Planned | #84 / #85 | LLM narrator + MCP tool exposure |
| P9 Streaming / intraday | 📋 Planned | #87 | O(1) streaming indicators, hourly/minute |
| Docs | ✅ Shipped | #86 | This notebook + supporting per-tool specs |

### Roadmap at a glance

```mermaid
gantt
    title techtrade phase shipping timeline (PRD §18)
    dateFormat  YYYY-MM-DD
    axisFormat  %b %Y

    section Core engine
    P0 Foundations               :done, p0, 2025-11-01, 20d
    P1 Segment screener          :done, p1, after p0, 15d
    P2 Indicator engine          :done, p2, after p1, 20d
    P3 Confluence engine         :done, p3, after p2, 20d
    P4 Rules + orders + fills    :done, p4, after p3, 20d
    P5 Recommendation + Excel    :done, p5, after p4, 20d

    section Validation + tuning
    P6 Validation bridge (#82)   :done, p6, 2026-06-01, 2026-06-21
    P7 Tuning (#83)              :done, p7, 2026-06-21, 2026-06-25

    section Data-pipeline
    #89 SEC 13F CUSIP index      :done, d1, 2026-05-15, 20d
    #93 OpenFIGI CUSIP-ticker    :done, d2, after d1, 15d
    #97 ETF holdings issuer tier :done, d3, after d2, 12d
    #99 SEC N-PORT ingest        :active, d4, 2026-06-25, 30d

    section Planned
    P8 Agent / MCP  (#84 / #85)  :p8, 2026-07-15, 45d
    P9 Streaming / intraday (#87):p9, after p8, 60d
    Docs (#86) - this notebook   :done, docs, 2026-06-01, 2026-06-30
```

### 6.1 Data-pipeline upgrades (free fallback tiers)

Three issues built out free-tier authoritative-data fallbacks so the
notebook doesn't depend on a paid FMP plan for every cell:

| Issue | Status | What landed |
|---|---|---|
| **#89** SEC bulk 13F CUSIP index | ✅ Shipped | `Tools/ingest_sec_13f.py` + read helpers (used by §1.6 and the `fmp_cached` `_try_sec_13f` tier shown in §3.1b) |
| **#93** OpenFIGI [CUSIP](https://www.investopedia.com/terms/c/cusipnumber.asp) → ticker resolver | ✅ Shipped | `Tools/enrich_cusip_figi.py` + `openbb_sec/utils/openfigi.py` (long-tail tickers for issuer-file holdings) |
| **#97** ETF holdings free fallback (SSGA half) | ✅ Shipped | `_try_issuer` tier in `fmp_cached/etf_holdings.py`; 11 SPDR ETFs covered by issuer-file even when FMP returns 402 |
| **#99** SEC N-PORT ingest | 🚧 In progress | 4 of 9 tasks merged on `trading_technicals` (T1 `sec_http`, T2 `nport_index`, T3 `nport_parser`, T4 `bulk_url_discovery`); T5 ingest-CLI, T6 wire `_try_nport`, T7 OpenFIGI ISIN ext, T8 enrich ISIN, T9 docs to come |

### 6.2 Try the working tuner

`obb.techtrade.tune` is now a real command. The shape (from
`docs/superpowers/specs/83-tuneta-adapter-tune-gated-validation.md`):

```python
# Tune one segment, gated by the §5 validation engine.
# Wall-clock: 5-15 minutes on a warm cache.
from openbb import obb
result = obb.techtrade.tune(segment="Information Technology")
report = result.results
print(report.verdict, report.persisted, report.candidate)
```

The design lock: only candidates whose [validation](https://www.investopedia.com/terms/b/backtesting.asp)
verdict comes back **`robust`** persist. `fragile` / `overfit` candidates are
returned for inspection but never reach the panel builder.

### 6.3 What this notebook will look like in v2

When #84 (narrator) lands, every cell's raw output is paired with an
LLM-generated plain-English summary. When #85 (MCP) lands, every
command in this notebook is also callable from an external LLM tool —
making the notebook itself redundant for headless use, but still the
best teaching surface for humans. When #99 finishes, the ETF holdings
fallback in §4 becomes "FMP → issuer-file → SEC N-PORT" with N-PORT
as the third authoritative tier.


## 7. This Notebook as a Verification Harness

One of the goals stated in the original brief: **this notebook is
our way of testing and verifying everything works fine**. Running
it end-to-end exercises every shipped surface:

- §1 verifies install + credentials.
- §3 exercises the standalone `Analysis` pipeline (no techtrade
  dependency).
- §4 exercises `obb.techtrade.{signals, plan, scan, export}`.
- §5 exercises `obb.techtrade.validate` against the in-tree
  `openbb-backtest`.
- §6 documents what's deliberately untested (because it's not yet
  shipped).

### 7.1 What a green run looks like

All cells below execute without `RuntimeError` or `ImportError`,
and produce non-empty results:

| Cell | Expected outcome |
|---|---|
| 1.3 environment | "Environment OK." prints; no missing required pkgs |
| 1.4 credentials | "Credentials OK." prints |
| 1.5 obb load | Lists `obb.techtrade.*` commands incl. `validate` |
| 3.1 Analysis | `Pipeline phases returned: ['p1'..'p7']` |
| 3.2 Phase-7 | Non-empty `action_label` and `composite_score` |
| 4.1 signals | At least one signal returned with finite score |
| 4.2 plan | One `TradePlan` with non-empty `orders` (or a flat plan with empty orders + clear note) |
| 4.3 scan | `len(plans) >= 1` |
| 4.4 export | A `.xlsx` file written with 6 sheets |
| 5.1 validate | `verdict in {'robust','fragile','overfit'}` + finite PBO/DSR |

### 7.2 Failure modes and what they mean

- **`ImportError: No module named 'openbb_techtrade'`** — your
  editable install didn't run from `.venv_win`. Re-do §1.1 step 4.
- **`KeyError: 'fmp_cached_api_key'`** or `401 Unauthorized` — bad
  or missing FMP key in `user_settings.json`. §1.2.
- **`429 Too Many Requests`** from FMP — you've burned through
  the free-tier daily limit. Wait an hour or upgrade your FMP
  plan.
- **Cell 4.3 returns 0 plans** — every sector's top mover scored
  below the entry threshold today. That's a real "no trade" day;
  not a bug. Try with `preset="mean_revert"` to test on a flatter
  tape.
- **Cell 5.1 hangs > 15 minutes** — the WFO fold loop is fetching
  cold OHLCV. Cancel, re-run §4 once to warm the cache, then
  retry §5.

### 7.3 When to evolve this into a multi-notebook series

The original brief mentioned **"A Developer Guide to Disciplined
Trading"** as the target. The natural decomposition once this POC
stabilizes:

| Future notebook | Focus | Triggers |
|---|---|---|
| `02-fundamentals-deep-dive.ipynb` | Each Analysis phase in detail | When P1-P7 questions exceed §3's depth |
| `03-confluence-internals.ipynb` | Indicator votes, preset weights, regime detection | When users ask "why this score?" |
| `04-tuning-and-validation.ipynb` | #83 `tune` + full WFO/CPCV walkthrough | When #83 T2-T7 ship |
| `05-paper-to-live.ipynb` | `simulate` -> live broker integration | When PRD P9 (live broker) lands |
| `06-agent-narrator-and-MCP.ipynb` | Hands-off LLM narrator + tool exposure | When #84 / #85 ship |

For now, **this single notebook is enough** — it covers every
shipped surface end-to-end. Iterate by editing
`notebooks/_build_notebook.py` and regenerating.


---

## Appendix A — File map

Where each piece of code lives in the repo:

| Path | What's there |
|---|---|
| `Analysis/stock_analysis.py` | 7-phase fundamentals pipeline (used in §3) |
| `openbb_platform/extensions/techtrade/` | The techtrade extension |
| `openbb_platform/extensions/techtrade/openbb_techtrade/engine/` | Indicators, confluence, rules, plan, scan |
| `openbb_platform/extensions/techtrade/openbb_techtrade/reporting/` | Excel export |
| `openbb_platform/extensions/techtrade/openbb_techtrade/validation/` | #82 backtest bridge + validate router |
| `openbb_platform/extensions/techtrade/openbb_techtrade/tuning/` | #83 tuneta adapter (T1 bootstrap shipped; T2-T7 pending) |
| `openbb_platform/extensions/backtest/` | In-tree `openbb-backtest` (WFO/CPCV/PBO/DSR engine) |
| `docs/Specs/TechnicalTrading-Engine-PRD.md` | The product spec everything traces back to |
| `docs/designs/quant_trading/` | One design doc per issue (#82, #83, #86) |
| `docs/superpowers/plans/` | TDD plans for in-flight features |
| `notebooks/01-foundations-techtrade-and-analysis.ipynb` | This notebook |

## Appendix B — Editing this notebook

**Edit cells directly in Jupyter Lab.** There is no separate builder
script; the `.ipynb` is the source of truth. The previous `_build_notebook.py`
was removed on 2026-06-27 — the indirection slowed iteration and meant
the rendered notebook drifted from the builder whenever someone added
a cell directly.

**Outputs are stripped** by the [`nbstripout`](https://github.com/kynan/nbstripout)
pre-commit hook, so the git diff of a notebook is the diff of its
Markdown + code only — execution counters and stdout never enter
history. This means PR reviews stay readable and merge conflicts
stay rare.

**Re-validate after big edits:**

```bash
.venv_win/Scripts/python.exe -c "import nbformat; nb=nbformat.read('notebooks/01-foundations-techtrade-and-analysis.ipynb', as_version=4); nbformat.validate(nb); print('OK')"
```

## Appendix C — Issue tracker

This fork uses **GitHub Issues** for human-facing tracking (the
#65/#74/#82/#83 numbers above) and **Beads (`bd`)** for
session-local sub-tasks (the `OpenBBTechnical-03x`-style IDs in
internal commits). Run `bd ready` from the repo root to see
what's available to work on; `bd prime` for the full command
reference.

---

*End of notebook 1. Series: "A Developer Guide to Disciplined
Trading". Maintained on the `trading_technicals` branch of
`prajoria/OpenBB`.*
